# Native head-VM recovery: short GCP pilot
Run one cell at a time: **settings → inventory/create → node setup → prepare → pilot → plot → collect**.
Use a fresh ray-dev kernel. All settings and calls below are ordinary editable Python, in the style of Selective_Replication_GCP. Helpers live in head_vm_loss/*.py; no subprocess wrapper or encoded code.

Actual head VM loss, Fixed-R OFF/ON, external Redis and a stable GCS proxy. This is a mechanism diagnostic, not proven application speedup. The old head's disk is not reused during recovery. Redis, proxy, driver and NFS survive on w4. See [the guide](head_vm_loss/NATIVE_PILOT.md). No cells have been executed by the assistant.

Fresh run: pull main, restart the kernel, leave EXISTING_STATE_FILE=None, and run sections 1–6 and 8–10 in order. Section 7 is for resuming only. Do not reuse RUN_ID after deleting VMs. The full five-row matrix can require another bounded run invocation; completed rows and failed outcomes are retained.


In [ ]:
import concurrent.futures
import json
import shlex
import subprocess
import sys
import time
import uuid
from pathlib import Path

PROJECT = "research-488322"
ZONE = "us-central1-a"
IMAGE = "ray-built-v1"
NETWORK = "default"
SUBNET = "default"
MACHINE_TYPE = "e2-standard-4"  # The current image check requires four CPUs.
DISK_GB = 100
PYTHON = "/opt/ray-env/ray-dev/bin/python"

# None creates fresh VMs. To reuse an existing cluster, enter its state.json path.
EXISTING_STATE_FILE = None  # Or Path("/absolute/path/to/state.json")
SUITE = "native-head-v5"
MAX_EXPERIMENT_MINUTES = 28  # Per run/resume invocation; cleanup reserves 90 seconds.
KEEPALIVE_INTERVAL_MS = 10000  # Same in every arm and on every Ray process.
KEEPALIVE_TIMEOUT_MS = 10000
TASKS = 16                  # 2..32
PAYLOAD_BYTES = 1024 * 1024  # 8 bytes..4 MiB
OBSERVATION_TIMEOUT_S = 300  # 60..300
# label, native protection, head VM loss, Redis/proxy enabled
OBSERVATIONS = [
    ("plain-healthy", False, False, False),
    ("off-healthy",   False, False, True),
    ("on-healthy",    True,  False, True),
    ("off-loss",      False, True,  True),
    ("on-loss",       True,  True,  True),
]

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "gossip_benchmarks").is_dir())
HELPERS = ROOT / "gossip_benchmarks/selective_replication"
NATIVE_HELPERS = ROOT / "gossip_benchmarks/head_vm_loss"
WORKDIR = ROOT / "ray_gcp_results"
sys.path.insert(0, str(HELPERS))
sys.path.insert(0, str(NATIVE_HELPERS))

if EXISTING_STATE_FILE is not None:
    STATE_FILE = Path(EXISTING_STATE_FILE).expanduser().resolve()
    state = json.loads(STATE_FILE.read_text())
    PROJECT, ZONE, RUN_ID = state["project"], state["zone"], state["run_id"]
    NAMES = state["names"]
else:
    if "RUN_ID" not in globals():
        RUN_ID = "rayexp-" + time.strftime("%m%d-%H%M%S") + "-" + uuid.uuid4().hex[:6]
    NAMES = [RUN_ID + "-head"] + [RUN_ID + f"-w{i}" for i in range(1, 5)]
    STATE_FILE = WORKDIR / RUN_ID / "state.json"
LOCAL = WORKDIR / RUN_ID
LOCAL.mkdir(parents=True, exist_ok=True)
HEAD, *WORKERS = NAMES
REGION = ZONE.rsplit("-", 1)[0]
print("VMs:", *NAMES, sep="\n  ")
print("State file:", STATE_FILE)


## 1. Small infrastructure helpers
Only infrastructure cells use these gcloud wrappers. They share a 28-minute setup budget starting with the first cloud command; pauses count. Preparation and each pilot invocation have separate bounded budgets, so initial provisioning plus the full matrix can take more than 30 minutes. No experiment starts automatically.


In [ ]:
def gcloud(*args, timeout=600):
    global EXPERIMENT_DEADLINE
    if "EXPERIMENT_DEADLINE" not in globals():
        EXPERIMENT_DEADLINE = time.monotonic() + 28 * 60
    if not globals().get("CLEANUP_MODE", False):
        remaining = EXPERIMENT_DEADLINE - time.monotonic()
        if remaining <= 0:
            raise TimeoutError("Infrastructure setup budget exhausted. Inspect saved state before continuing.")
        timeout = min(timeout, remaining)
    command = ["gcloud", *map(str, args), f"--project={PROJECT}", "--quiet"]
    print(shlex.join(command), flush=True)
    result = subprocess.run(command, text=True, capture_output=True, timeout=timeout)
    if result.returncode:
        raise RuntimeError(result.stderr + "\n" + result.stdout)
    return result.stdout


def ssh(name, command):
    return gcloud("compute", "ssh", name, f"--zone={ZONE}", "--tunnel-through-iap",
                  "--ssh-flag=-oConnectTimeout=15", f"--command={command}")


def scp(source, target):
    return gcloud("compute", "scp", source, target, f"--zone={ZONE}", "--tunnel-through-iap")


def run_parallel(function, items):
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as pool:
        return list(pool.map(function, items))


## 2. Inspect current instances — read only
Do not operate this cluster while another notebook is using it. Existing VMs must match the recorded ownership labels.


In [ ]:
inventory = json.loads(gcloud("compute", "instances", "list", "--format=json"))
for vm in inventory:
    if vm.get("labels", {}).get("ray-experiment"):
        print(vm["name"], vm["status"], vm["zone"].rsplit("/", 1)[-1])


## 3. Create fresh VMs, or skip for an existing state file
Uses the existing default VPC/subnet. No network is created. Names are recorded before allocation; after a partial creation failure inspect the state rather than silently creating a second cluster.


In [ ]:
from gcp.capabilities import stop_command

print("Failure command:", "gcloud", *stop_command(), "--no-graceful-shutdown")
if STATE_FILE.exists():
    print("Creation already recorded for this session. Continue with setup.", STATE_FILE)
else:
    image = json.loads(gcloud("compute", "images", "describe", IMAGE, "--format=json"))
    assert image["status"] == "READY"
    gcloud("compute", "networks", "describe", NETWORK, "--format=json")
    subnet = json.loads(gcloud("compute", "networks", "subnets", "describe", SUBNET,
                               f"--region={REGION}", "--format=json"))
    assert subnet["network"].rsplit("/", 1)[-1] == NETWORK, "Subnet belongs to a different VPC"
    # Record names before creation so partial failures can be cleaned up.
    STATE_FILE.write_text(json.dumps(dict(project=PROJECT, zone=ZONE, run_id=RUN_ID,
                                         names=NAMES, image=IMAGE, network=NETWORK, subnet=SUBNET), indent=2))
    gcloud("compute", "firewall-rules", "create", RUN_ID + "-internal",
           f"--network={NETWORK}", "--allow=tcp,udp,icmp",
           f"--source-tags={RUN_ID}", f"--target-tags={RUN_ID}")
    gcloud("compute", "firewall-rules", "create", RUN_ID + "-iap",
           f"--network={NETWORK}", "--allow=tcp:22",
           "--source-ranges=35.235.240.0/20", f"--target-tags={RUN_ID}")
    gcloud("compute", "instances", "create", *NAMES, f"--zone={ZONE}",
           f"--machine-type={MACHINE_TYPE}", f"--image={IMAGE}", f"--image-project={PROJECT}",
           f"--boot-disk-size={max(DISK_GB, int(image['diskSizeGb']))}GB",
           "--boot-disk-type=pd-balanced", "--provisioning-model=STANDARD",
           f"--subnet={SUBNET}", f"--tags={RUN_ID}", f"--labels=ray-experiment={RUN_ID}",
           "--metadata=enable-oslogin=FALSE", "--no-service-account", "--no-scopes", timeout=900)


## 4. Get IPs and wait for SSH
For a resume after the last head-loss observation, skip creation, node setup and preparation. Reconnect to the saved preparation in section 7 instead.


In [ ]:
nodes = []
for name in NAMES:
    vm = json.loads(gcloud("compute", "instances", "describe", name, f"--zone={ZONE}", "--format=json"))
    assert vm.get("labels", {}).get("ray-experiment") == RUN_ID, name
    assert vm["status"] == "RUNNING", (name, vm["status"])
    nodes.append(dict(name=name, ip=vm["networkInterfaces"][0]["networkIP"], head=name == HEAD))
HEAD_IP = nodes[0]["ip"]
print(json.dumps(nodes, indent=2))


def wait_ssh(name):
    for attempt in range(30):
        try:
            ssh(name, "true")
            return
        except (RuntimeError, subprocess.TimeoutExpired):
            if attempt == 29:
                raise
            time.sleep(5)


wait_ssh(HEAD)  # Initialize gcloud's local SSH key before parallel connections.
run_parallel(wait_ssh, WORKERS)


## 5. Install the visible node helper
The image already contains Ray/PyTorch. This sets up the experiment account and node identity files. It does not create head-hosted NFS: surviving storage is prepared on w4 next.


In [ ]:
def setup_node(node):
    name = node["name"]
    ssh(name, """set -e
sudo apt-get update -qq
sudo DEBIAN_FRONTEND=noninteractive apt-get -o DPkg::Lock::Timeout=120 install -y -qq nfs-common
id raybench >/dev/null 2>&1 || sudo useradd --uid 25001 --create-home --home-dir /var/lib/raybench --shell /bin/bash raybench
test "$(id -u raybench)" = 25001
sudo mkdir -p /opt/ray-gcp /srv/ray-gcp
mountpoint -q /srv/ray-gcp || sudo chown raybench:raybench /srv/ray-gcp
""")
    path = LOCAL / (name + ".json")
    path.write_text(json.dumps(dict(node, nodes=nodes, head_ip=HEAD_IP, run_id=RUN_ID)))
    scp(path, name + ":/tmp/ray-node.json")
    scp(HELPERS / "gcp/bootstrap/node_control.py", name + ":/tmp/ray-node-control.py")
    ssh(name, """set -e
sudo install -o root -g root -m 644 /tmp/ray-node.json /etc/ray-gcp.json
sudo install -o root -g root -m 644 /tmp/ray-node-control.py /opt/ray-gcp/node_control.py
""")


run_parallel(setup_node, nodes)


## 6. Prepare the native runner — initial setup only
Moves evidence storage to w4, installs Redis/socat and uploads the native helpers. No workload is run here. Skip this cell on an ordinary resume. Source/image mismatch or an existing suite lock requires inspection.

Includes the NFS export-directory fix and authenticated Redis startup. Gate traffic uses HTTP on w4 port 8765, which the experiment's internal firewall rule permits. Preparation uploads the helpers; pre-workload checks verify the gate is reachable from both executors.


In [ ]:
from native_pilot import NativePilot

cloud = NativePilot(STATE_FILE, WORKDIR / "head-vm-loss")
cloud.prepare()
print("Prepared:", cloud.local)


## 7. Reconnect after restarting the kernel — optional
Skip if section 6 just succeeded. To resume, set EXISTING_STATE_FILE and the same SUITE/settings above, run the settings cell, then this cell. It reads local preparation state and checks CLI support without resetting the cluster. The run call checks instance ownership and helper/settings fingerprints.


In [ ]:
RESUME_PREPARED = False
if RESUME_PREPARED:
    from native_pilot import NativePilot
    from gcp.capabilities import stop_command

    cloud = NativePilot(STATE_FILE, WORKDIR / "head-vm-loss")
    environment = json.loads((cloud.local / "environment.json").read_text())
    cloud.nodes = environment["nodes"]
    cloud.stop_args = stop_command()
    cloud.ready = True
    print("Using saved preparation:", cloud.local)


## 8. Run or resume the editable matrix
Change the settings before starting a new suite. Resumes require the same workload and matrix; changing them requires a new SUITE. Failed/timeout observations are retained. A 28-minute invocation may stop before finishing all observations; rerun this cell to continue. Fixed-R remains R=2, W=2.

The matched 10-second keepalive settings test the slow owner-detection hypothesis; they do not guarantee recovery. Original tasks use an off-head HTTP gate instead of NFS existence checks. Native log tails and final GCS state are saved for every observation before reset. The successful-fault check requires replay evidence for all remaining outputs.

Native SSH/SCP calls retry only explicit IAP 4003 / port 22 connection-creation failures, at most three attempts within the existing timeout. Persistent failures are retained and stop the run; see the optional diagnostic cell below.


In [ ]:
print("Workload:", TASKS, "tasks,", PAYLOAD_BYTES, "bytes per output")
for observation in OBSERVATIONS:
    print(observation)

summary = cloud.run_pilot(
    SUITE,
    budget_s=MAX_EXPERIMENT_MINUTES * 60,
    tasks=TASKS,
    payload_bytes=PAYLOAD_BYTES,
    timeout_s=OBSERVATION_TIMEOUT_S,
    observations=OBSERVATIONS,
    keepalive_interval_ms=KEEPALIVE_INTERVAL_MS,
    keepalive_timeout_ms=KEEPALIVE_TIMEOUT_MS,
)
print("Finished matrix:", summary.get("finished_matrix", False))
print("Cleanup errors:", summary.get("cleanup_errors", []))


## 9. Inspect saved progress — no cloud calls
Failures before injection are setup failures, not evidence of failed recovery.


In [ ]:
REPORT = cloud.local / SUITE / "native-report.json"
report = json.loads(REPORT.read_text())
print("Recorded:", len(report["samples"]), "/", report["planned_observations"])
for row in report["samples"]:
    result = row.get("result", {})
    print(row["label"], row["status"], "correct:", result.get("correct", False),
          "elapsed:", result.get("elapsed_s"))
    if row.get("error") or result.get("error"):
        print("  ", row.get("error") or result.get("error"))
print("Results:", REPORT.parent)

for row in report["samples"]:
    if row.get("fault"):
        print(row["label"], "replays:", row.get("result", {}).get("reexecutions"),
              "head stayed off:", row.get("original_head_remained_off"),
              "final GCS:", row.get("gcs_at_observation_end"))


## Plot saved results — run this cell alone
Set RESULTS_DIR to a saved suite folder, or leave None when cloud and SUITE exist in this kernel. No preparation, SSH or reruns. Failed outcomes stay visible. Points only, no bars or bottom annotations. Timing changes from one replicate are preliminary.


In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt

RESULTS_DIR = None  # Or Path("/absolute/path/to/native-head-v5")
if RESULTS_DIR is None:
    if "cloud" not in globals() or "SUITE" not in globals():
        raise ValueError("Set RESULTS_DIR to your saved suite folder")
    RESULTS_DIR = cloud.local / SUITE
REPORT_PATH = Path(RESULTS_DIR).expanduser() / "native-report.json"
report = json.loads(REPORT_PATH.read_text())
rows = report.get("samples", [])
by_label = {row["label"]: row for row in rows}

for baseline, candidate, label in [
    ("plain-healthy", "off-healthy", "Redis/proxy runtime overhead"),
    ("off-healthy", "on-healthy", "Native Fixed-R overhead"),
    ("plain-healthy", "on-healthy", "Combined enablement overhead"),
]:
    a = by_label.get(baseline, {}).get("result", {})
    b = by_label.get(candidate, {}).get("result", {})
    av = a.get("elapsed_excluding_supervisor_gate_s")
    bv = b.get("elapsed_excluding_supervisor_gate_s")
    if a.get("correct") and b.get("correct") and av and bv is not None:
        print(f"{label}: {100 * (bv / av - 1):+.2f}% ({av:.3f}s -> {bv:.3f}s)")
    else:
        print(label + ": unavailable; needs two correct healthy results")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for i, row in enumerate(rows):
    result = row.get("result", {})
    if result.get("correct"):
        axes[0].scatter(result["elapsed_s"], i, color="tab:blue")
    else:
        axes[0].text(.02, i, row["status"], color="tab:red",
                     transform=axes[0].get_yaxis_transform(), va="center")
axes[0].set_yticks(range(len(rows)), [r["label"] for r in rows])
axes[0].invert_yaxis()
axes[0].set_title("Correct job completions")
axes[0].set_xlabel("Driver elapsed time including gate (s)")

faults = [r for r in rows if r.get("fault")]
for i, row in enumerate(faults):
    value = row.get("failure_to_resumption_upper_bound_s")
    if value is not None:
        axes[1].scatter(value, i, color="tab:orange")
    else:
        axes[1].text(.02, i, "No verified resumption", color="tab:red",
                     transform=axes[1].get_yaxis_transform(), va="center")
axes[1].set_yticks(range(len(faults)), [r["label"] for r in faults])
axes[1].invert_yaxis()
axes[1].set_title("Observed resumption upper bounds")
axes[1].set_xlabel("Time from stop request (s)")
fig.tight_layout()
output = REPORT_PATH.parent / "native-head-pilot.png"
fig.savefig(output, dpi=160)
plt.show()
print("Saved:", output)


## 10. Collect evidence before deleting VMs
Copies native log tails, task attempt evidence and Redis state from w4. Keep the local native-report.json as well. This cell does not rerun workloads. If collection fails, keep w4 and its disk until you have the evidence.


In [ ]:
print("Local results:", cloud.local / SUITE)
DOWNLOAD_EVIDENCE = True
if DOWNLOAD_EVIDENCE:
    evidence_dir = cloud.download(SUITE)
    print("Upload:", cloud.local / SUITE / "native-report.json")
    print("Upload:", evidence_dir / "evidence.tar.gz")


## Diagnose an IAP/SSH failure — optional
After a connection error, restart the kernel and run the settings cell with EXISTING_STATE_FILE pointing to this cluster's saved state.json. Set DIAGNOSE_CONNECTIONS=True below. This cell checks VM identity/status, SSH reachability, support services and the suite lock; it changes no cloud state.

An IAP 4003 / port 22 error is a controller transport failure, not evidence of Ray recovery failure. The native runner now retries that specific connection-creation error up to three times within its original command/session budget. Other errors and timeouts are not blindly retried. Events are saved in native-transport.jsonl and each observation's transport_events.

Keep the old suite/report. If all VMs are RUNNING, SSH is reachable, cleanup succeeded, and no suite lock remains, set SUITE="native-head-v5", run section 6 to upload the updated helpers, then section 8. Do not use section 7 to bypass the helper update. If cleanup errors or a lock remain, inspect them before proceeding; this notebook does not automatically remove a lock. You do not need to recreate a healthy cluster solely because a connection failed.


In [ ]:
# Run independently after the settings cell. Does not start/stop VMs or Ray.
DIAGNOSE_CONNECTIONS = False
PREVIOUS_SUITE = "native-head-v4"

if DIAGNOSE_CONNECTIONS:
    from native_pilot import NativePilot

    diagnostic_cloud = NativePilot(STATE_FILE, WORKDIR / "head-vm-loss")
    environment_path = diagnostic_cloud.local / "environment.json"
    if environment_path.exists():
        diagnostic_cloud.nodes = json.loads(environment_path.read_text())["nodes"]
    previous_report = diagnostic_cloud.local / PREVIOUS_SUITE / "native-report.json"
    if previous_report.exists():
        previous = json.loads(previous_report.read_text())
        print("Previous cleanup errors:", previous.get("cleanup_errors", "not recorded"))
        for row in previous.get("samples", []):
            print(row["label"], row["status"])
    for name in diagnostic_cloud.names:
        try:
            vm = diagnostic_cloud.describe(name)
            print(name, vm["status"])
            if vm["status"] == "RUNNING":
                probe = diagnostic_cloud.ssh(name, "true", check=False, timeout=30)
                print("  SSH:", "reachable" if probe.returncode == 0 else probe.stderr[-2000:])
            if name == diagnostic_cloud.support and vm["status"] == "RUNNING":
                probe = diagnostic_cloud.ssh(name, """uptime
systemctl is-active ssh
systemctl list-units 'ray-headloss-*' 'ray-native-*' --state=active,activating --no-legend
if test -d /var/lib/raybench/ray-headloss-lock; then
    printf 'Suite lock is present; inspect cleanup before proceeding.\\n'
else
    printf 'No suite lock.\\n'
fi
""", check=False, timeout=30)
                print(probe.stdout, probe.stderr)
        except Exception as exc:
            print(name, str(exc))


## 11. Optional VM stop
Stops only the five recorded, ownership-checked VMs. Disks/evidence remain allocated; no deletion. Download wanted results first. Do not stop a cluster with a live controller or unresolved cleanup errors.


In [ ]:
STOP_VMS = False
if STOP_VMS:
    if cloud.ssh(cloud.support, "test ! -d /var/lib/raybench/ray-headloss-lock", check=False).returncode:
        raise RuntimeError("Suite lock exists; inspect the controller before stopping VMs")
    for name in cloud.names:
        vm = cloud.describe(name)
        if vm["status"] == "RUNNING":
            cloud.gc("compute", "instances", "stop", name, "--zone=" + cloud.zone, timeout=300)
else:
    print("VM stop skipped. Running VMs and persistent disks remain billable.")
